# Phase 3 Experiments - Phase B: Knowledge Distillation Only

This notebook runs:
- Run 2.1: KD1 - XLM-RoBERTa -> SahajBERT (eval_fold=3, matching T1)
- Run 2.2: KD2 - XLM-RoBERTa -> BanglaBERT-small (eval_fold=3, matching T1)
- Run 2.3: KD3 - BanglaBERT -> SahajBERT (eval_fold=2, matching T2)
- Run 2.4: KD4 - BanglaBERT -> BanglaBERT-small (eval_fold=2, matching T2)

**Then uploads all 4 KD models to HuggingFace for reuse in Scenarios 3 & 4!**

**IMPORTANT**: KD models use teacher's eval_fold for fair comparison!
- KD1, KD2 (Teacher T1): eval_fold=3
- KD3, KD4 (Teacher T2): eval_fold=2

**Estimated Time**: 6-8 hours

**Dependencies**: None (can run immediately)

**CRITICAL**: Phase C+D depend on this notebook completing and uploading models!

**Total Runs**: 4

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q iterative-stratification scikit-learn pandas numpy tqdm
!pip install -q huggingface_hub

In [ ]:
# Setup logging
import sys
from datetime import datetime

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger("/kaggle/working/experiment_log.txt")
print(f"Experiment started at: {datetime.now()}")
print("Phase B: Knowledge Distillation Only")

In [ ]:
# Clone repository
!git clone https://github.com/SaifSiddique009/kd_pruning_quantization_framework_for_nlp.git
%cd kd_pruning_quantization_framework_for_nlp
!git checkout phase3-comprehensive-experiments
!git log -1 --oneline

In [ ]:
# Verify dataset
import os
DATASET_PATH = "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv"

if os.path.exists(DATASET_PATH):
    import pandas as pd
    df = pd.read_csv(DATASET_PATH)
    print(f"Dataset loaded: {len(df)} samples")
else:
    print("ERROR: Dataset not found!")

In [ ]:
# IMPORTANT: Login to HuggingFace for model upload
# You need to set your HF token as a Kaggle secret named "HF_TOKEN"
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    !huggingface-cli login --token $HF_TOKEN
    print("HuggingFace login successful!")
except:
    print("WARNING: HF_TOKEN not found in Kaggle secrets.")
    print("You'll need to manually upload models or set up the secret.")
    print("Go to: Add-ons > Secrets > Add secret named 'HF_TOKEN'")

---
## Scenario 2: Knowledge Distillation Only (4 runs)

### KD Combinations:
| KD ID | Teacher | Student | Description |
|-------|---------|---------|-------------|
| KD1 | T1 (XLM-RoBERTa) | RS1 (SahajBERT) | Cross-lingual teacher |
| KD2 | T1 (XLM-RoBERTa) | RS2 (BanglaBERT-small) | Cross-lingual to small |
| KD3 | T2 (BanglaBERT) | RS1 (SahajBERT) | Bangla-specific teacher |
| KD4 | T2 (BanglaBERT) | RS2 (BanglaBERT-small) | Same-family distillation |

---

### Run 2.1: KD1 - T1 -> RS1 (XLM-RoBERTa -> SahajBERT) (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario2/KD1_T1_RS1

### Run 2.2: KD2 - T1 -> RS2 (XLM-RoBERTa -> BanglaBERT-small) (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "csebuetnlp/banglabert_small" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario2/KD2_T1_RS2

### Run 2.3: KD3 - T2 -> RS1 (BanglaBERT -> SahajBERT) (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-sagor-bangla-bert-base" \
    --student_path "neuropark/sahajBERT" \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario2/KD3_T2_RS1

### Run 2.4: KD4 - T2 -> RS2 (BanglaBERT -> BanglaBERT-small) (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-sagor-bangla-bert-base" \
    --student_path "csebuetnlp/banglabert_small" \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario2/KD4_T2_RS2

---
## Check KD Results Before Upload
---

In [ ]:
import os
import json
from datetime import datetime

print(f"\n{'='*70}")
print(f"KD EXPERIMENT STATUS - {datetime.now()}")
print(f"{'='*70}\n")

kd_experiments = [
    ("2.1 KD1 (T1->RS1)", "./results/scenario2/KD1_T1_RS1", 3),
    ("2.2 KD2 (T1->RS2)", "./results/scenario2/KD2_T1_RS2", 3),
    ("2.3 KD3 (T2->RS1)", "./results/scenario2/KD3_T2_RS1", 2),
    ("2.4 KD4 (T2->RS2)", "./results/scenario2/KD4_T2_RS2", 2),
]

ready_for_upload = []

print(f"{'Experiment':<25} {'Fold':<6} {'F1 Weighted':<12} {'F1 Macro':<12} {'Status'}")
print("-" * 70)

for name, output_dir, fold in kd_experiments:
    json_path = os.path.join(output_dir, "results_final.json")
    model_path = os.path.join(output_dir, "model_final_hf")
    
    if os.path.exists(json_path) and os.path.exists(model_path):
        with open(json_path) as f:
            data = json.load(f)
        
        # Get final metrics
        if isinstance(data, list):
            final_metrics = data[-1]
        else:
            final_metrics = data
        
        f1_weighted = final_metrics.get('f1_weighted', 'N/A')
        f1_macro = final_metrics.get('f1_macro', 'N/A')
        
        print(f"{name:<25} {fold:<6} {f1_weighted:<12.4f} {f1_macro:<12.4f} READY")
        ready_for_upload.append(name)
    elif os.path.exists(json_path):
        print(f"{name:<25} {fold:<6} {'N/A':<12} {'N/A':<12} PARTIAL")
    else:
        print(f"{name:<25} {fold:<6} {'N/A':<12} {'N/A':<12} FAILED")

print("-" * 70)
print(f"\nReady for upload: {len(ready_for_upload)}/4 models")
print(f"{'='*70}")

---
## Upload KD Models to HuggingFace

**CRITICAL STEP**: This uploads all 4 KD models so they can be reused in Phase C+D!

---

In [ ]:
# Upload all KD models using the upload script
!python upload_kd_models.py --author_name Saif-Siddique

In [ ]:
# Alternative: Manual upload if the script fails
# Uncomment and run if needed

# from huggingface_hub import HfApi, create_repo
# 
# api = HfApi()
# author = "Saif-Siddique"
# 
# kd_uploads = [
#     ("KD1", "kd1-xlmroberta-to-sahajbert", "./results/scenario2/KD1_T1_RS1/model_final_hf"),
#     ("KD2", "kd2-xlmroberta-to-banglabert-small", "./results/scenario2/KD2_T1_RS2/model_final_hf"),
#     ("KD3", "kd3-banglabert-to-sahajbert", "./results/scenario2/KD3_T2_RS1/model_final_hf"),
#     ("KD4", "kd4-banglabert-to-banglabert-small", "./results/scenario2/KD4_T2_RS2/model_final_hf"),
# ]
# 
# for kd_id, name, path in kd_uploads:
#     if os.path.exists(path):
#         repo_name = f"{author}/bangla-cyberbully-{name}"
#         print(f"Uploading {kd_id} to {repo_name}...")
#         create_repo(repo_name, exist_ok=True)
#         api.upload_folder(folder_path=path, repo_id=repo_name)
#         print(f"SUCCESS: {kd_id}")
#     else:
#         print(f"SKIP: {kd_id} - path not found")

---
## Final Status & KD Model Paths
---

In [ ]:
# Print the HuggingFace paths for Phase C+D
!python upload_kd_models.py --author_name Saif-Siddique --print_paths

In [ ]:
# Save results locally
!cp -r ./results /kaggle/working/
!python aggregate_results.py --results_dir ./results --output /kaggle/working/phase_b_summary.csv --format all

In [ ]:
print(f"\n{'='*60}")
print("PHASE B COMPLETE!")
print(f"{'='*60}")
print("\nKD Models uploaded to HuggingFace:")
print("  - Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert")
print("  - Saif-Siddique/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small")
print("  - Saif-Siddique/bangla-cyberbully-kd3-banglabert-to-sahajbert")
print("  - Saif-Siddique/bangla-cyberbully-kd4-banglabert-to-banglabert-small")
print("\nPhase C+D can now use these models with --pipeline prune_only!")
print(f"\nCompleted at: {datetime.now()}")